## Simulate BERT data

In [1]:
import os
import torch
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from mpstemmer import MPStemmer

import gensim
from gensim import corpora
from gensim.utils import simple_preprocess
from pprint import pprint

from transformers import BertTokenizer
from nltk.tokenize import RegexpTokenizer

os.environ["CUDA_VISIBLE_DEVICES"] = "1"
print("Total GPU:", torch.cuda.device_count())
print("Current GPU:", torch.cuda.get_device_name(torch.cuda.current_device()))

Total GPU: 1
Current GPU: NVIDIA RTX A5000


In [2]:
path = "../bert_data/id-p2"

train_files = []
val_files = []
test_files = []
for file in os.listdir(path):
    if "bert.pt" in file and "train" in file:
        train_files.append(path + "/" + file)
    elif "bert.pt" in file and "valid" in file:
        val_files.append(path + "/" + file)
    elif "bert.pt" in file and "test" in file:
        test_files.append(path + "/" + file)

train_files = sorted(train_files)
val_files = sorted(val_files)
test_files = sorted(test_files)

In [3]:
train_docs = []

i = 0
for file in train_files:
    print(f"Loading data train {i}...")
    bert_data = torch.load(file)
    for data in bert_data:
        src = " ".join(data['src_txt'])
        # src = data['src_txt']
        train_docs.append(src)
    i = i + 1

Loading data train 0...
Loading data train 1...
Loading data train 2...
Loading data train 3...
Loading data train 4...
Loading data train 5...
Loading data train 6...
Loading data train 7...
Loading data train 8...
Loading data train 9...
Loading data train 10...
Loading data train 11...
Loading data train 12...
Loading data train 13...
Loading data train 14...
Loading data train 15...
Loading data train 16...
Loading data train 17...
Loading data train 18...
Loading data train 19...


In [4]:
val_docs = []

i = 0
for file in val_files:
    print(f"Loading data val {i}...")
    bert_data = torch.load(file)
    for data in bert_data:
        src = " ".join(data['src_txt'])
        # src = data['src_txt']
        val_docs.append(src)
    i = i + 1

Loading data val 0...
Loading data val 1...
Loading data val 2...


In [5]:
test_docs = []

i = 0
for file in test_files:
    print(f"Loading data test {i}...")
    bert_data = torch.load(file)
    for data in bert_data:
        src = " ".join(data['src_txt'])
        # src = data['src_txt']
        test_docs.append(src)
    i = i + 1

Loading data test 0...
Loading data test 1...
Loading data test 2...


In [129]:
print(train_docs[0])

['dilaporkan', 'dua', 'orang', 'terluka', 'cukup', 'serius', 'sementara', 'sisanya', 'sudah', 'diperbolehkan', 'pulang', 'setelah', 'mendapatkan', 'perawatan', 'video', 'gajah', 'yang', 'mengamuk', 'itu', 'kontan', 'viral', 'di', 'media', 'sosial', 'si', 'gajah', 'berlari', 'tak', 'tentu', 'arah', 'menabrak', 'serta', 'menginjak', 'sebagian', 'peserta', 'upacara', 'simak', 'juga', 'belum', 'diketahui', 'penyebab', 'gajah', 'tersebut', 'tiba', 'tiba', 'mengamuk', 'diduga', 'gajah', 'itu', 'kaget', 'oleh', 'sesuatu', 'di', 'antara', 'para', 'peserta', 'dan', 'pengunjung', 'media', 'setempat', 'melaporkan', 'gajah', 'lain', 'yang', 'juga', 'mengamuk', 'di', 'prosesi', 'berbeda', 'gajah', 'hias', 'merupakan', 'daya', 'tarik', 'tersendiri', 'dalam', 'upacara', 'keagamaan', 'di', 'sri', 'lanka', 'bagi', 'warga', 'sri', 'lanka', 'memiliki', 'gajah', 'adalah', 'simbol', 'status', 'beberapa', 'kuil', 'di', 'sri', 'lanka', 'juga', 'memiliki', 'gajah', 'untuk', 'keperluan', 'upacara']


In [7]:
with open('stopwords.txt') as file:
    stopwords = [line.rstrip() for line in file]
    
def preprocess(docs):
    # Split the documents into tokens.
    tokenizer = RegexpTokenizer(r'\w+')
    for idx in range(len(docs)):
        docs[idx] = docs[idx].lower()  # Convert to lowercase.
        docs[idx] = tokenizer.tokenize(docs[idx])  # Split into words.

    # Remove numbers, but not words that contain numbers.
    docs = [[token for token in doc if not token.isnumeric()] for doc in docs]
    
    # Remove words that are only one character.
    docs = [[token for token in doc if len(token) > 1] for doc in docs]
    
    # Tokenize and remove stopwords
    docs = [[token for token in doc if token not in stopwords] for doc in docs]
    
    return docs

In [8]:
proc_train_docs = preprocess(train_docs)
proc_val_docs = preprocess(val_docs)
proc_test_docs = preprocess(test_docs)

print("Saving docs...")
torch.save(proc_train_docs, "./lda/proc_train_doc.pt")
torch.save(proc_val_docs, "./lda/proc_val_doc.pt")
torch.save(proc_test_docs, "./lda/proc_test_doc.pt")

Saving docs...


In [11]:
proc_train_docs = torch.load("./lda/proc_train_doc.pt")
proc_val_docs = torch.load("./lda/proc_val_doc.pt")
proc_test_docs = torch.load("./lda/proc_test_doc.pt")

### LDA

In [14]:
# Create a dictionary representation of the documents.
dictionary = corpora.Dictionary(proc_train_docs)

# Filter out words that occur less than 20 documents, or more than 50% of the documents.
dictionary.filter_extremes(no_below=20, no_above=0.5)

In [15]:
# Bag-of-words representation of the documents.
corpus = [dictionary.doc2bow(doc) for doc in proc_train_docs]
val_bow = [dictionary.doc2bow(doc) for doc in proc_val_docs]
test_bow = [dictionary.doc2bow(doc) for doc in proc_test_docs]

In [16]:
print('Number of unique tokens: %d' % len(dictionary))
print('Number of documents: %d' % len(corpus))

Number of unique tokens: 19801
Number of documents: 38207


In [17]:
import logging
logging.basicConfig(format='%(asctime)s : %(levelname)s : %(message)s',
                   level=logging.DEBUG,
                   filename='lda_model.log')

In [18]:
# Train LDA model.
from gensim.models import LdaModel

# Set training parameters.
num_topics = 250
chunksize = 5000
passes = 1
iterations = 400
eval_every = None  # Don't evaluate model perplexity, takes too much time.

# Make an index to word dictionary.
temp = dictionary[0]  # This is only to "load" the dictionary.
id2word = dictionary.id2token

model = LdaModel(
    corpus=corpus,
    id2word=id2word,
    chunksize=chunksize,
    alpha='auto',
    eta='auto',
    iterations=iterations,
    num_topics=num_topics,
    passes=passes,
    eval_every=eval_every
)

In [80]:
len(model.print_topics(num_topics=100, num_words=10))

50

### Topic-Token Distribution

In [81]:
tokenizer = BertTokenizer.from_pretrained("indobenchmark/indobert-base-p2")
vocab = tokenizer.get_vocab()

In [82]:
# map_topic_id_to_index = {}

lda_topics = model.print_topics(num_topics=num_topics)
topics_raw = {}
for i in range(len(lda_topics)):
    topic_id, topic_components = lda_topics[i]
    # map_topic_id_to_index[topic_id] = i
    
    topic_components = topic_components.split(" + ")
    new_components = []
    for comp in topic_components:
        score, word = comp.split("*")
        word = word.replace('"', '')
        new_components.append((word, float(score)))
    topics_raw[i] = new_components

In [83]:
# Convert words into tokens
# Ex: {0: [('korea', 0.1), ('utara', 0.2)]}
# --> {0: [('ko', 0.1), ('##rea', 0.1), ('utara', 0.2)]}
topics_tokenized = {}
for topic, words in topics_raw.items():
    if topic < 0:
        continue
    topics_tokenized[topic] = []    
    for word, score in words:
        tokens = tokenizer.tokenize(word)
        for tok in tokens:
            topics_tokenized[topic].append((tok, score))

# Clean topics that have same tokens inside it and assign the highest score
# Ex: {0: [('ko', 0.1), ('##rea', 0.1), ('utara', 0.2), ('ko', 0.01)]}
# --> {0: [('ko', 0.1), ('##rea', 0.1), ('utara', 0.2)]}
topics_tokenized_clean = {}
for topic, tokens in topics_tokenized.items():
    seen_tok = []
    topics_tokenized_clean[topic] = []
    for tok, score in tokens:
        if tok in seen_tok:
            continue
        else:
            seen_tok.append(tok)
            topics_tokenized_clean[topic].append((tok, score))

In [93]:
# Let us round the maximum to 50
MAX_TOKENS = 20

# The original ones: 5 x 30521
# [[0., 0., 0.,  ..., 0., 0., 0.],
# [0., 0., 0.,  ..., 0., 0., 0.],
# [0., 0., 0.,  ..., 0., 0., 0.],
# [0., 0., 0.,  ..., 0., 0., 0.],
# [0., 0., 0.,  ..., 0., 0., 0.]]

# We make it smaller like this: 5 x 50
# [[(token, token_score), (token, token_score), ..., MAX_TOKENS],
# [(token, token_score), (token, token_score), ..., MAX_TOKENS],
# [(token, token_score), (token, token_score), ..., MAX_TOKENS],
# [(token, token_score), (token, token_score), ..., MAX_TOKENS],
# [(token, token_score), (token, token_score), ..., MAX_TOKENS]]

In [85]:
topics_T = []
for topic in topics_tokenized_clean:
    li = []
    for token_tuple in topics_tokenized_clean[topic]:
        token, score = token_tuple
        li.append((token, score))

    if len(li) < MAX_TOKENS:
        add_li = [('[PAD]', 0)for _ in range(MAX_TOKENS - len(li))]
        li = li + add_li
        
    topics_T.append(li)

print(topics_T[:2])

[[('nigeria', 0.05), ('uang', 0.043), ('haram', 0.037), ('bok', 0.028), ('##o', 0.028), ('mug', 0.015), ('##abe', 0.015), ('zi', 0.014), ('##mb', 0.014), ('##ab', 0.014), ('##we', 0.014), ('juventus', 0.012), ('sunat', 0.01), ('cat', 0.008), ('##alunya', 0.008), ('kamer', 0.007), ('##un', 0.007), ('[PAD]', 0), ('[PAD]', 0), ('[PAD]', 0), ('[PAD]', 0), ('[PAD]', 0), ('[PAD]', 0), ('[PAD]', 0), ('[PAD]', 0), ('[PAD]', 0), ('[PAD]', 0), ('[PAD]', 0), ('[PAD]', 0), ('[PAD]', 0), ('[PAD]', 0), ('[PAD]', 0), ('[PAD]', 0), ('[PAD]', 0), ('[PAD]', 0), ('[PAD]', 0), ('[PAD]', 0), ('[PAD]', 0), ('[PAD]', 0), ('[PAD]', 0), ('[PAD]', 0), ('[PAD]', 0), ('[PAD]', 0), ('[PAD]', 0), ('[PAD]', 0), ('[PAD]', 0), ('[PAD]', 0), ('[PAD]', 0), ('[PAD]', 0), ('[PAD]', 0)], [('hewan', 0.015), ('hutan', 0.011), ('spesies', 0.008), ('liar', 0.008), ('daging', 0.008), ('manusia', 0.007), ('alam', 0.006), ('binatang', 0.006), ('lingkungan', 0.006), ('ikan', 0.006), ('[PAD]', 0), ('[PAD]', 0), ('[PAD]', 0), ('[PAD

In [86]:
# Approximate topic distribution for each doc
train_topic_distr = []
for bow in corpus:
    distr = model.get_document_topics(bow, minimum_probability=0)
    distr.sort(key=lambda x: x[1], reverse=True)
    train_topic_distr.append(distr[:5])
    

In [87]:
print(train_topic_distr[2])

[(41, 0.52995884), (15, 0.26751223), (42, 0.13923024), (22, 0.05319773), (31, 0.00030444545)]


## Not Scored

In [95]:
# Assign each document with its related topics (train)
# Assign topic distribution over words
# The size is K x V where K is topics related to the document and V is vocab size
# There are 2 methods: with scoring and without scoring
# Scoring: the contribution score of topic to the doc will be multiplied to the topic distribution
is_scoring = False
path = "../bert_data/id-p2"

train_files = []

for file in os.listdir(path):
    if "bert.pt" in file and "train" in file:
        train_files.append(path + "/" + file)

train_files = sorted(train_files)

data_index = 0
for file in train_files:
    filename = file.split("/")[-1]

    # Loop only for train files
    if "bert.pt" in file and "train" in file:
        print(f"Processing {filename}...")
        bert_data = torch.load(file)  # inside each files contains 2000 data

        # Loop through each data
        for i in range(len(bert_data)):
            # Find related topics
            # Loop through topic_distr to get corresponding topic for each doc
            d = train_topic_distr[i][:5]
            
            bert_data[i]['topic_dist'] = []
            distribution_over_words = []
            
            # We make it smaller like this: K x MAX_TOKENS
            # [{token: token_score, token: token_score, ..., MAX_TOKENS},
            # {token: token_score, token: token_score, ..., MAX_TOKENS},
            # {token: token_score, token: token_score, ..., MAX_TOKENS}]
            
            if len(d) > 0:
                # Assign topic distribution over words 
                for item in d:
                    topic_id = item[0]
                    topic_score = item[1]
                    
                    if is_scoring:
                        topics_T_scored = []
                        for token_tuple in topics_T[topic_id]:
                            new_tuple = (token_tuple[0], token_tuple[1] * topic_score) # multiply by the contribution score of the topic
                            topics_T_scored.append(new_tuple)
                        distribution_over_words.append(topics_T_scored) 
                    else:
                        distribution_over_words.append(topics_T[topic_id])
                        
            # Set the dimension to be equal across documents
            empty_topic = [('[PAD]', 0) for _ in range(MAX_TOKENS)]
            distribution_over_words = distribution_over_words + [empty_topic] * (5 - len(distribution_over_words))

            bert_data[i]['topic_dist'] = distribution_over_words
            
            data_index = data_index + 1
        
        torch.save(bert_data, f"./lda/topics/{filename}")

Processing xlsum.train.0.bert.pt...
Processing xlsum.train.1.bert.pt...
Processing xlsum.train.10.bert.pt...
Processing xlsum.train.11.bert.pt...
Processing xlsum.train.12.bert.pt...
Processing xlsum.train.13.bert.pt...
Processing xlsum.train.14.bert.pt...
Processing xlsum.train.15.bert.pt...
Processing xlsum.train.16.bert.pt...
Processing xlsum.train.17.bert.pt...
Processing xlsum.train.18.bert.pt...
Processing xlsum.train.19.bert.pt...
Processing xlsum.train.2.bert.pt...
Processing xlsum.train.3.bert.pt...
Processing xlsum.train.4.bert.pt...
Processing xlsum.train.5.bert.pt...
Processing xlsum.train.6.bert.pt...
Processing xlsum.train.7.bert.pt...
Processing xlsum.train.8.bert.pt...
Processing xlsum.train.9.bert.pt...
